In [1]:

from vaccbopti.classes.person import Person
from vaccbopti.classes.params import Params
# from vaccbopti.classes.timesteps import People
import random as rd
import numpy as np

params = Params.instance()



In [2]:
class BoosterAdmin:
    def __init__(self):
        self.vacc_list = []
        self.vacc_indices = []


    def update_susceptibility(self, vaccine_choice, person):
        ''' 
        An individual's status is changed to 'vacc' (vaccinated), and their immunity_time
        is set to 0 (if given the old vaccine, immunity_time_exvacc, if new vaccine, immunity_time_newvacc).
        This implies they will not be included in the list of people eligible for vaccination.
        vaccine_choice: 'old_vacc' (existing vaccine) or 'new_vacc' (updated vaccine when it becomes available).
        '''
        # access calc_suceptibility
        if vaccine_choice == 'old_vacc':
            person.immunity_time_exvacc = 0
            person.vacc_status = 'vacc'
        elif vaccine_choice == 'new_vacc':
            person.immunity_time_newvacc = 0
            person.vacc_status = 'vacc'
    
    #need to change 'person.' to the correct word 
    # we now have a unique ID for each person in People
    def vaccine_administration(self, People, vaccine_choice, direction, age_targets):
        ''' 
        In this function, a list is created of all eligible individuals to be vaccinated (those who are
        symptomatic, hospitalised, dead, vaccinated, or not 'eligible for vaccination' (person.vacc_status='unvacc')). 
        Uneligible refers to the 20% of the population that would not be vaccinated for various reasons.
        The list is randomised and resorted according to the vaccination strategy (param 'direction'):
        - descending: re-organise the list into descending age groups (from old to young)
        - ascending: re-organise the list into ascending age groups (young to old)
        - random: no re-organising.
        A sub-population of the list can be selected (param 'age_targets'):
        - mid-old: age groups 50-75+
        - mid-young: age groups 0-49
        - everyone: all age groups
        The to-be-vaccinated individuals are randomly selected, from 1000 per day to the remaining
        amount of individuals in the list available.
        Finally, the vacc_status is changed using the update_susceptibility function.
        '''
        self.vacc_list = [p for p in People 
                          if p.status not in ['symptomatic', 'hospitalised', 'dead'] 
                          and p.vacc_status == 'unvacc']
        #shuffle the list
        rd.shuffle(self.vacc_list)

        # then resort it into age groups but with the people shuffled
        # descend goes from old to young people
        if direction == 'descend':
            self.vacc_list.sort(key=lambda p: params.age_groups.index(p.age_group), reverse=True)
        # ascend goes from young to old people
        elif direction == 'ascend':
            self.vacc_list.sort(key=lambda p: params.age_groups.index(p.age_group))
        # in case we don't want to sort by age
        elif direction == 'random':
            self.vacc_list = self.vacc_list

        # decide if you want to use all, 50+ (old groups), or 49- (young groups)
        if age_targets == 'everyone':
            self.vacc_list = self.vacc_list
        elif age_targets == 'mid-young':
            self.vacc_list = [p for p in self.vacc_list
                 if p.age_group in params.young_groups]
        elif age_targets == 'mid-old':
            self.vacc_list = [p for p in self.vacc_list
                 if p.age_group in params.old_groups]

        # select either 1000 or the length of the remaining unvaccinated people
        limit = min(10, len(self.vacc_list))
        self.vacc_indices = [self.vacc_list[i].id for i in range(limit)]
        print(len(self.vacc_indices))
        print(f' the people to be vaccinated: {self.vacc_indices}')
        
        # 'give vaccine' and update status
        for p in People:
            if p.id in self.vacc_indices:
                print(f' p id to change: {p.id}')
                print(f' age group of p: {p.age_group}')
                # change old_vacc -> vaccine_choice
                self.update_susceptibility(vaccine_choice, p)
    
    def vacc_strat_1(self, People):
        '''
        This vaccine strategy vaccinates everyone starting at the oldest age group and descending,
        not taking into account the availability of the updated vaccine.

        '''
        self.vaccine_administration(People, vaccine_choice='old_vacc',direction='descend',age_targets='everyone')

    def vacc_strat_2(self, People, t, t_newvacc_avail):
        '''
        This vaccine strategy vaccinates everyone starting at the oldest age group and descending,
        when the updated vaccine becomes available.
        '''
        if t > t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='new_vacc', direction='descend',age_targets='everyone')
    
    def vacc_strat_3 (self, People, t, t_newvacc_avail):
        '''
        The third strategy starts vaccinating with the old vaccine from the oldest age groups and descending 
        (from 75+ down), until the new vaccine becomes available. At this point, the new vaccine starting at 
        the middle age groups is prioritised in a descending way (from 49 down). When all the updated vaccines
        have been administered, the old vaccination is continued in the older age groups.

        t : time (in days)
        t_newvacc_avail : time when updated vaccine becomes available
        '''
        if t < t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='old_vacc',direction='descend',age_targets='mid-old')
        elif t > t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='new_vacc', direction='descend',age_targets='mid-young')
        elif len(self.vacc_list) == 0:
            self.vaccine_administration(People, vaccine_choice='old_vacc',direction='descend',age_targets='mid-old')
    
    def vacc_strat_4(self, People, t, t_newvacc_avail):
        '''
        The fourth strategy starts vaccinating with the old vaccine to the youngest age groups ascending (0+ up),
        and switches to vaccinating from the middle age groups up (50+ and up) until all have been vaccinated with the 
        updated vaccine. It then switches back to vaccinating the remaining individuals in the young age groups with the 
        old vaccine.
        
        t : time (in days)
        t_newvacc_avail : time when updated vaccine becomes available

        '''
        if t < t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='old_vacc',direction='ascend',age_targets='mid-young')
        elif t > t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='new_vacc', direction='ascend', age_targets='mid-old')
        elif len(self.vacc_list) == 0:
            self.vaccine_administration(People, vaccine_choice='old_vacc', direction='ascend', age_targets='mid-young')
    
    def vacc_strat_5(self, People):
        ''' 
        The old vaccine is administered randomnly to anyone within the population.
        '''
        self.vaccine_administration(People, vaccine_choice='old_vacc',direction='random',age_targets='everyone')
    
    def vacc_strat_6(self, People,t, t_newvacc_avail):
        ''' 
        The updated vaccine is administered randomnly to anyone within the population when it becomes available.

        t : time (in days)
        t_newvacc_avail : time when updated vaccine becomes available

        '''
        if t > t_newvacc_avail:
            self.vaccine_administration(People, vaccine_choice='new_vacc',direction='random',age_targets='everyone')




In [8]:
for a in range(20):
    test = BoosterAdmin().vacc_strat_6(People, t=a, t_newvacc_avail=15)
    print(f' day {a} completed')




 day 0 completed
 day 1 completed
 day 2 completed
 day 3 completed
 day 4 completed
 day 5 completed
 day 6 completed
 day 7 completed
 day 8 completed
 day 9 completed
 day 10 completed
 day 11 completed
 day 12 completed
 day 13 completed
 day 14 completed
 day 15 completed
10
 the people to be vaccinated: [228, 345, 248, 335, 210, 313, 371, 355, 282, 353]
 p id to change: 210
 age group of p: 60-64
 p id to change: 228
 age group of p: 70-74
 p id to change: 248
 age group of p: 30-34
 p id to change: 282
 age group of p: 35-39
 p id to change: 313
 age group of p: 15-19
 p id to change: 335
 age group of p: 25-29
 p id to change: 345
 age group of p: 20-24
 p id to change: 353
 age group of p: 0-4
 p id to change: 355
 age group of p: 65-69
 p id to change: 371
 age group of p: 40-44
 day 16 completed
10
 the people to be vaccinated: [337, 234, 394, 296, 365, 325, 224, 331, 277, 368]
 p id to change: 224
 age group of p: 50-54
 p id to change: 234
 age group of p: 35-39
 p id to c

In [6]:

statuses = ['susceptible', 'exposed', 'asymptomatic', 'symptomatic', 'hospitalised', 'dead']
vacc_statuses = ['unvacc', 'ineligible']
People = [Person() for p in range(200)]
for person in People:
    person.age_group = str(np.random.choice(params.age_groups))
    person.status = str(np.random.choice(statuses))
    person.vacc_status = str(np.random.choice(vacc_statuses))


In [7]:
count = sum(1 for p in People 
            if p.status not in ['symptomatic', 'hospitalised', 'dead'] 
            and p.vacc_status == 'unvacc')
print(f'Eligible for vaccination: {count}')

Eligible for vaccination: 39


In [62]:
testadmin = BoosterAdmin().vaccine_administration(People, 'old_vacc')

[]
[]
[]
 the people to be vaccinated: []


In [50]:

for p in People:
    print(f' for {p.vacc_status} the age group is {p.age_group}')

 for unvacc the age group is 25-29
 for ineligible the age group is 20-24
 for ineligible the age group is 10-14
 for unvacc the age group is 60-64
 for unvacc the age group is 60-64
 for vacc the age group is 15-19
 for ineligible the age group is 40-44
 for ineligible the age group is 5-9
 for unvacc the age group is 75+
 for ineligible the age group is 60-64
 for ineligible the age group is 70-74
 for vacc the age group is 65-69
 for ineligible the age group is 45-49
 for vacc the age group is 10-14
 for vacc the age group is 60-64
 for ineligible the age group is 65-69
 for unvacc the age group is 55-59
 for ineligible the age group is 15-19
 for vacc the age group is 75+
 for ineligible the age group is 15-19
 for unvacc the age group is 0-4
 for vacc the age group is 15-19
 for ineligible the age group is 70-74
 for unvacc the age group is 40-44
 for vacc the age group is 15-19
 for ineligible the age group is 25-29
 for unvacc the age group is 10-14
 for vacc the age group is 20

In [3]:
#### testing the ouput
# Class to store data output
import pandas as pd
from vaccbopti.classes.params import Params
params = Params.instance()


class Output:
    def __init__(self):
        self.tracked_status = ['symptomatic', 'hospitalised', 'dead']
        self.vacc_states = ['vacc', 'unvacc']
        self.statusDF = None
    
    def setdf(self):
        #pull column names
        columns = ['t']
        for age_group in params.age_groups:
            for status in self.tracked_status:
                for vacc_state in self.vacc_states:
                    columns.append(f'{age_group}_{status}_{vacc_state}')
        self.statusDF = pd.DataFrame(columns=columns)
    
    def initialise_df(self):
        self.setdf()
    
    def append_daily_nbs(self, t, People=People):
        row = {'t': t}
        for age_group in params.age_groups:
            for status in self.tracked_status:
                for vacc_state in self.vacc_states:
                    count = sum(1 for p in People 
                            if p.age_group == age_group 
                            and p.status == status 
                            and p.vacc_status == vacc_state)
                    print(f' for {age_group} {status} {vacc_state} the sum is {count}')
                    row[f'{age_group}_{status}_{vacc_state}'] = count
        # append to df        
        self.statusDF = pd.concat([self.statusDF, pd.DataFrame([row])], ignore_index=True)


In [2]:

statuses = ['susceptible', 'exposed', 'asymptomatic', 'symptomatic', 'hospitalised', 'dead']
vacc_statuses = ['unvacc', 'ineligible']
People = [Person() for p in range(500)]
for person in People:
    person.age_group = str(np.random.choice(params.age_groups))
    person.status = str(np.random.choice(statuses))
    person.vacc_status = str(np.random.choice(vacc_statuses))

In [ ]:
output = Output()
output.setdf()
for t in range(5):
    output.append_daily_nbs(t=t)
print(output.statusDF)

 for 0-4 symptomatic vacc the sum is 0
 for 0-4 symptomatic unvacc the sum is 4
 for 0-4 hospitalised vacc the sum is 0
 for 0-4 hospitalised unvacc the sum is 3
 for 0-4 dead vacc the sum is 0
 for 0-4 dead unvacc the sum is 6
 for 5-9 symptomatic vacc the sum is 0
 for 5-9 symptomatic unvacc the sum is 1
 for 5-9 hospitalised vacc the sum is 0
 for 5-9 hospitalised unvacc the sum is 0
 for 5-9 dead vacc the sum is 0
 for 5-9 dead unvacc the sum is 1
 for 10-14 symptomatic vacc the sum is 0
 for 10-14 symptomatic unvacc the sum is 4
 for 10-14 hospitalised vacc the sum is 0
 for 10-14 hospitalised unvacc the sum is 2
 for 10-14 dead vacc the sum is 0
 for 10-14 dead unvacc the sum is 3
 for 15-19 symptomatic vacc the sum is 0
 for 15-19 symptomatic unvacc the sum is 1
 for 15-19 hospitalised vacc the sum is 0
 for 15-19 hospitalised unvacc the sum is 3
 for 15-19 dead vacc the sum is 0
 for 15-19 dead unvacc the sum is 1
 for 20-24 symptomatic vacc the sum is 0
 for 20-24 symptomatic 

AttributeError: 'Output' object has no attribute 'stausDF'

In [5]:
print(output.statusDF)

   t 0-4_symptomatic_vacc 0-4_symptomatic_unvacc 0-4_hospitalised_vacc  \
0  0                    0                      4                     0   
1  1                    0                      4                     0   
2  2                    0                      4                     0   
3  3                    0                      4                     0   
4  4                    0                      4                     0   

  0-4_hospitalised_unvacc 0-4_dead_vacc 0-4_dead_unvacc 5-9_symptomatic_vacc  \
0                       3             0               6                    0   
1                       3             0               6                    0   
2                       3             0               6                    0   
3                       3             0               6                    0   
4                       3             0               6                    0   

  5-9_symptomatic_unvacc 5-9_hospitalised_vacc  ... 70-74_hospitalised_vac